# Lesson04. 机器狗派对！编排你的专属舞步

**教学主题：** 用代码创造复杂的连续动作。

**核心目标：** 掌握 `for` 循环与函数（`function`）的使用。

**课程安排：**

- **前30分钟（新工具）：** 将一套"拜年动作"（连续点头10次+招手）打包成一个叫做 `new_year_greeting()` 的指令。

- **后90分钟（创意编舞）：**
  - **设计舞步：** 学习控制身体姿态的指令（摇摆、扭动），设计2-3个基本舞步。
  - **串联与重复：** 舞步用函数打包，再用循环重复播放，创作一段30秒的机器狗舞蹈。
  - 播放音乐，全场的Go2一起开派对！

## 4.1 导入依赖并初始化客户端

In [ ]:
import time  # 时间模块，用于控制延时
import sys   # 系统模块

# 导入宇树SDK通信和运动控制模块
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道并创建运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()

## 4.2 使用for循环编排动作队列

`for` 循环可以让一段代码重复执行多次，非常适合让机器狗重复做某个动作。

In [ ]:
# 连续旋转：循环4次，每次以1.0 rad/s的角速度旋转，持续1秒
for i in range(4):
    sport_client.Move(0, 0, 1.0)  # 原地逆时针旋转
    time.sleep(1)  # 等待1秒再发送下一条指令

In [ ]:
# 连续执行动作：先做2次Scrape（刨地），再做2次Content（撒娇）
for i in range(2):
    sport_client.Scrape()   # 刨地动作
    # sport_client.Stretch()  # 伸展动作（已注释，可取消注释尝试）
    # time.sleep(4)

for i in range(2):
    sport_client.Content()  # 撒娇/开心动作

In [ ]:
# 使用if-else条件判断交替执行不同动作
# i % 2 == 0 表示偶数次执行Scrape，奇数次执行Content
for i in range(4):
    if i % 2 == 0:
        print("刨地动作")
        sport_client.Scrape()
    else:
        print("撒娇动作")
        sport_client.Content()

## 4.3 多动作串联

将多个动作按顺序串联起来，封装成一个函数，实现一套完整的表演流程。

In [ ]:
import time
import sys
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道和运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


def run_robot_actions():
    """按顺序执行一套完整的机器人动作表演"""
    # ---- 第一部分：坐下与起立 ----
    ret = sport_client.Sit()           # 坐下
    print(f"坐下动作执行结果: {ret}")
    time.sleep(3)                       # 等待动作完成

    ret = sport_client.RiseSit()        # 从坐姿起立
    print(f"起立动作执行结果: {ret}")

    # ---- 第二部分：交替表演刨地和撒娇 ----
    for i in range(4):
        if i % 2 == 0:
            print("刨地动作")
            sport_client.Scrape()
        else:
            print("撒娇动作")
            sport_client.Content()
        time.sleep(2)                   # 每个动作间隔2秒

    # ---- 第三部分：高难度动作展示 ----
    sport_client.StandUp()              # 站立
    print("站立")
    time.sleep(3)

    sport_client.FrontPounce()          # 前扑
    print("前扑")
    time.sleep(3)

    sport_client.FrontFlip()            # 前空翻
    print("前空翻")
    time.sleep(3)

    sport_client.Scrape()               # 刨地
    print("刨地")
    time.sleep(3)

    # ---- 第四部分：舞蹈表演 ----
    sport_client.Dance1()               # 舞蹈动作1
    print("舞蹈1")
    time.sleep(3)

    sport_client.Dance2()               # 舞蹈动作2
    print("舞蹈2")
    time.sleep(3)

    # 表演结束，回到站立状态
    sport_client.StandUp()
    time.sleep(2)


if __name__ == "__main__":
    try:
        run_robot_actions()
        print("所有动作执行完成！")
    except Exception as e:
        print(f"执行过程中出现错误: {e}")
        # 出错时尝试让机器人恢复站立
        sport_client.StandUp()
        time.sleep(2)

## 4.4 随机动作组合

从动作库中随机挑选指定数量的动作执行，每次运行效果都不一样！

In [ ]:
import time
import random  # 随机数模块，用于随机选择动作
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道和运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


def run_random_actions(num_actions):
    """
    从动作库中随机选择指定数量的不重复动作并依次执行。
    
    Args:
        num_actions (int): 要执行的动作数量
    """
    # 定义动作列表：(动作名称, 动作函数, 动作后等待时间)
    actions = [
        ("坐下",   sport_client.Sit, 3),
        ("起立",   sport_client.RiseSit, 0),
        ("刨地",   sport_client.Scrape, 3),
        ("撒娇",   sport_client.Content, 2),
        ("站立",   sport_client.StandUp, 3),
        ("前扑",   sport_client.FrontPounce, 3),
        ("前空翻", sport_client.FrontFlip, 3),
        ("舞蹈1",  sport_client.Dance1, 3),
        ("舞蹈2",  sport_client.Dance2, 3),
    ]
    
    # 参数检查
    if num_actions <= 0:
        raise ValueError("动作数量必须为正整数")
    if num_actions > len(actions):
        raise ValueError(f"动作数量不能超过动作库总数（{len(actions)}个）")
    
    # 使用random.sample随机选择不重复的动作
    selected_actions = random.sample(actions, num_actions)
    
    # 依次执行选中的动作
    for action_name, action_func, delay in selected_actions:
        try:
            print(f"执行动作: {action_name}")
            ret = action_func()
            print(f"  返回值: {ret}")
            if delay > 0:
                time.sleep(delay)
        except Exception as e:
            print(f"  执行动作 {action_name} 时出错: {e}")
            time.sleep(1)
    
    print(f"\n{num_actions}个随机动作全部执行完成！")


if __name__ == "__main__":
    # 设置要随机执行的动作数量（可自行修改）
    CUSTOM_NUM_ACTIONS = 5
    
    try:
        run_random_actions(CUSTOM_NUM_ACTIONS)
    except ValueError as ve:
        print(f"参数错误: {ve}")
    except Exception as e:
        print(f"程序执行出错: {e}")
        # 出错时尝试恢复站立
        try:
            sport_client.StandUp()
            time.sleep(3)
        except:
            pass